In [2]:
# ============================================================
# BLOCK 1: Environment Setup
# CRATTT Clean Implementation
# Dada Victor Damilare | MRES7015 | University of Greater Manchester
# ============================================================

# --- 1.1 Install Required Libraries ---
!pip install imagecorruptions -q
!pip install wandb -q
!pip install pycocotools -q
!pip install ultralytics -q

# --- 1.2 Core Imports ---
import os
import sys
import random
import json
import glob
import numpy as np
import torch
import skimage
import skimage.filters
import imagecorruptions.corruptions as cor_mod
import imagecorruptions
import wandb
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# --- 1.3 Reproducibility: Fix All Seeds ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
print(f"✅ Seeds fixed: {SEED}")

# --- 1.4 Surgical Patch: NumPy 2.0 + scikit-image compatibility ---
# imagecorruptions uses the deprecated 'multichannel' kwarg removed in
# scikit-image 0.19+. This patch redirects it to the new 'channel_axis'.
if not hasattr(skimage.filters.gaussian, '_is_patched'):
    _real_gaussian = skimage.filters.gaussian

    def _patched_gaussian(*args, **kwargs):
        if 'multichannel' in kwargs:
            val = kwargs.pop('multichannel')
            kwargs['channel_axis'] = -1 if val else None
        return _real_gaussian(*args, **kwargs)

    _patched_gaussian._is_patched = True
    skimage.filters.gaussian = _patched_gaussian
    cor_mod.gaussian = _patched_gaussian
    print("✅ scikit-image patch applied")
else:
    print("✅ scikit-image patch already active")

# --- 1.5 Log Library Versions for Reproducibility ---
print("\n--- Library Versions ---")
print(f"Python:            {sys.version.split()[0]}")
print(f"PyTorch:           {torch.__version__}")
print(f"NumPy:             {np.__version__}")
print(f"scikit-image:      {skimage.__version__}")

import importlib.metadata
try:
    ic_version = importlib.metadata.version("imagecorruptions")
except importlib.metadata.PackageNotFoundError:
    ic_version = "installed (version unknown)"
print(f"imagecorruptions:  {ic_version}")

# --- 1.6 Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n✅ Device: {device}")
if device.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# --- 1.7 Secure API Connections ---
user_secrets = UserSecretsClient()
print("\n--- API Connections ---")

try:
    wb_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wb_key, relogin=True)
    print("✅ W&B: Connected")
except Exception as e:
    print(f"⚠️  W&B: Not connected — {e}")

try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✅ HuggingFace: Connected")
except Exception as e:
    print(f"❌ HuggingFace: Failed — {e}")

# --- 1.8 Create Output Directories ---
DIRS = {
    "results":  "/kaggle/working/results",
    "figures":  "/kaggle/working/figures",
    "checkpoints": "/kaggle/working/checkpoints",
    "tables":   "/kaggle/working/tables"
}
for name, path in DIRS.items():
    os.makedirs(path, exist_ok=True)

print(f"\n✅ Output directories created:")
for name, path in DIRS.items():
    print(f"   {name}: {path}")

print("\n" + "="*50)
print("BLOCK 1 COMPLETE — Environment ready")
print("="*50)

✅ Seeds fixed: 42
✅ scikit-image patch applied

--- Library Versions ---
Python:            3.12.12
PyTorch:           2.10.0+cu128
NumPy:             2.0.2
scikit-image:      0.25.2
imagecorruptions:  1.1.2

✅ Device: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB

--- API Connections ---


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vd1res (vd1res-university-of-bolton) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ W&B: Connected
✅ HuggingFace: Connected

✅ Output directories created:
   results: /kaggle/working/results
   figures: /kaggle/working/figures
   checkpoints: /kaggle/working/checkpoints
   tables: /kaggle/working/tables

BLOCK 1 COMPLETE — Environment ready


In [3]:
# ============================================================
# BLOCK 2: Model Loading
# Loads GroundingDINO, YOLO-World, and CLIP Oracle once.
# Never reload models in subsequent blocks.
# ============================================================

import torch
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from transformers import CLIPProcessor, CLIPModel
from ultralytics import YOLOWorld

print("Loading models to device:", device)
print("="*50)

# --- 2.1 GroundingDINO ---
print("\n[1/3] Loading GroundingDINO-tiny...")
DINO_MODEL_ID = "IDEA-Research/grounding-dino-tiny"

dino_processor = AutoProcessor.from_pretrained(
    DINO_MODEL_ID, token=hf_token
)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    DINO_MODEL_ID, token=hf_token
).to(device)
dino_model.eval()

dino_params = sum(p.numel() for p in dino_model.parameters())
print(f"✅ GroundingDINO-tiny loaded")
print(f"   Parameters: {dino_params:,}")
print(f"   VRAM used so far: "
      f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")

# --- 2.2 YOLO-World ---
print("\n[2/3] Loading YOLO-World-large...")

# CPU handoff pattern: set classes on CPU to avoid CUDA text-encoding bug
yolo_model = YOLOWorld('yolov8l-world.pt')
yolo_model.to('cpu')

# We will set the full class list in Block 3 once COCO_MAP is defined
# For now just verify it loads
print(f"✅ YOLO-World-large loaded")
print(f"   VRAM used so far: "
      f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")

# --- 2.3 CLIP Oracle ---
print("\n[3/3] Loading CLIP Oracle (ViT-B/32)...")
CLIP_MODEL_ID = "openai/clip-vit-base-patch32"

clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device)
clip_model.eval()

clip_params = sum(p.numel() for p in clip_model.parameters())
print(f"✅ CLIP Oracle loaded")
print(f"   Parameters: {clip_params:,}")
print(f"   VRAM used so far: "
      f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")

# --- 2.4 VRAM Summary ---
vram_allocated = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
vram_free = vram_total - vram_allocated

print("\n--- VRAM Summary ---")
print(f"Allocated: {vram_allocated:.2f} GB")
print(f"Free:      {vram_free:.2f} GB")
print(f"Total:     {vram_total:.2f} GB")

if vram_free < 4.0:
    print("⚠️  WARNING: Less than 4GB free. "
          "Consider reducing NUM_IMAGES in Block 4.")
else:
    print("✅ VRAM headroom is sufficient for inference.")

print("\n" + "="*50)
print("BLOCK 2 COMPLETE — All models loaded")
print("="*50)

Loading models to device: cuda

[1/3] Loading GroundingDINO-tiny...


The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/990 [00:00<?, ?it/s]

✅ GroundingDINO-tiny loaded
   Parameters: 172,249,090
   VRAM used so far: 0.69 GB

[2/3] Loading YOLO-World-large...
✅ YOLO-World-large loaded
   VRAM used so far: 0.69 GB

[3/3] Loading CLIP Oracle (ViT-B/32)...


The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ CLIP Oracle loaded
   Parameters: 151,277,313
   VRAM used so far: 1.30 GB

--- VRAM Summary ---
Allocated: 1.30 GB
Free:      14.34 GB
Total:     15.64 GB
✅ VRAM headroom is sufficient for inference.

BLOCK 2 COMPLETE — All models loaded


In [4]:
# ============================================================
# BLOCK 3: Global Constants
# Single source of truth for all experimental parameters.
# Never hardcode these values in subsequent blocks.
# ============================================================

import torch
import torch.nn.functional as F

# --- 3.1 COCO Category Mapping ---
# Verified against official COCO 2017 annotation file
COCO_MAP = {
    'person': 1, 'bicycle': 2, 'car': 3, 'motorcycle': 4,
    'airplane': 5, 'bus': 6, 'train': 7, 'truck': 8, 'boat': 9,
    'traffic light': 10, 'fire hydrant': 11, 'stop sign': 13,
    'parking meter': 14, 'bench': 15, 'bird': 16, 'cat': 17,
    'dog': 18, 'horse': 19, 'sheep': 20, 'cow': 21,
    'elephant': 22, 'bear': 23, 'zebra': 24, 'giraffe': 25,
    'backpack': 27, 'umbrella': 28, 'handbag': 31, 'tie': 32,
    'suitcase': 33, 'frisbee': 34, 'skis': 35, 'snowboard': 36,
    'sports ball': 37, 'kite': 38, 'baseball bat': 39,
    'baseball glove': 40, 'skateboard': 41, 'surfboard': 42,
    'tennis racket': 43, 'bottle': 44, 'wine glass': 46,
    'cup': 47, 'fork': 48, 'knife': 49, 'spoon': 50, 'bowl': 51,
    'banana': 52, 'apple': 53, 'sandwich': 54, 'orange': 55,
    'broccoli': 56, 'carrot': 57, 'hot dog': 58, 'pizza': 59,
    'donut': 60, 'cake': 61, 'chair': 62, 'couch': 63,
    'potted plant': 64, 'bed': 65, 'dining table': 67,
    'toilet': 70, 'tv': 72, 'laptop': 73, 'mouse': 74,
    'remote': 75, 'keyboard': 76, 'cell phone': 77,
    'microwave': 78, 'oven': 79, 'toaster': 80, 'sink': 81,
    'refrigerator': 82, 'book': 84, 'clock': 85, 'vase': 86,
    'scissors': 87, 'teddy bear': 88, 'hair drier': 89,
    'toothbrush': 90
}

COCO_CLASSES = list(COCO_MAP.keys())  # 80 classes
print(f"✅ COCO_MAP loaded: {len(COCO_MAP)} categories")

# --- 3.2 Verify COCO_MAP Against Ground Truth ---
# This catches any ID mismatches before they silently corrupt mAP scores
from pycocotools.coco import COCO

ANN_PATH = ("/kaggle/input/datasets/awsaf49/"
            "coco-2017-dataset/coco2017/annotations/"
            "instances_val2017.json")
IMAGE_DIR = ("/kaggle/input/datasets/awsaf49/"
             "coco-2017-dataset/coco2017/val2017")

coco_gt = COCO(ANN_PATH)
official_cats = {
    cat['name']: cat['id']
    for cat in coco_gt.loadCats(coco_gt.getCatIds())
}

mismatches = []
for name, our_id in COCO_MAP.items():
    official_id = official_cats.get(name)
    if official_id is None:
        mismatches.append(f"  NOT FOUND in COCO: '{name}'")
    elif official_id != our_id:
        mismatches.append(
            f"  MISMATCH: '{name}' → ours={our_id}, "
            f"official={official_id}"
        )

if mismatches:
    print("⚠️  COCO_MAP mismatches detected:")
    for m in mismatches:
        print(m)
else:
    print("✅ COCO_MAP verified: all 80 IDs match official annotations")

# --- 3.3 Text Prompts ---
# GroundingDINO expects dot-separated class names
DINO_TEXT_PROMPT = " . ".join(COCO_CLASSES) + " ."

# YOLO-World expects plain class names
# Set classes here using the CPU handoff pattern
yolo_model.to('cpu')
yolo_model.set_classes(COCO_CLASSES)
yolo_model.to(device)
print("✅ YOLO-World classes set: 80 COCO categories")


# --- 3.4 CLIP Per-Class Text Embeddings ---
print("\nGenerating per-class CLIP text embeddings...")

with torch.no_grad():
    embeddings = []
    
    for class_name in COCO_CLASSES:
        inputs = clip_processor(
            text=[class_name],
            return_tensors="pt",
            padding=True
        ).to(device)
        
        # Bypass get_text_features entirely
        # Call text_model directly, extract last hidden state,
        # then apply projection manually
        text_out = clip_model.text_model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )
        
        # Pooled output is the [EOS] token representation
        # Shape: [1, hidden_dim] = [1, 512]
        pooled = text_out.pooler_output
        
        # Apply the learned projection to get final embedding
        projected = clip_model.text_projection(pooled)
        embeddings.append(projected)
    
    clip_text_features = torch.cat(embeddings, dim=0)
    clip_text_features = F.normalize(clip_text_features, p=2, dim=-1)

print(f"✅ CLIP text embeddings: {clip_text_features.shape}")
print(f"   Expected: torch.Size([80, 512])")

person_idx = COCO_CLASSES.index('person')
car_idx    = COCO_CLASSES.index('car')
chair_idx  = COCO_CLASSES.index('chair')
dog_idx    = COCO_CLASSES.index('dog')

sim_person_car  = (clip_text_features[person_idx] @ clip_text_features[car_idx]).item()
sim_person_dog  = (clip_text_features[person_idx] @ clip_text_features[dog_idx]).item()
sim_car_chair   = (clip_text_features[car_idx] @ clip_text_features[chair_idx]).item()

print(f"\n   Similarity checks (all must be < 0.98):")
print(f"   person vs car:   {sim_person_car:.4f}")
print(f"   person vs dog:   {sim_person_dog:.4f}")
print(f"   car vs chair:    {sim_car_chair:.4f}")

all_distinct = all(
    s < 0.98 for s in [sim_person_car, sim_person_dog, sim_car_chair]
)
print(f"\n   {'✅ All embeddings are distinct' if all_distinct else '❌ Still identical — check CLIP model'}")


# --- 3.5 ImageNet-C Corruption Protocol ---
CORRUPTION_CATEGORIES = {
    "Noise":   ["gaussian_noise", "shot_noise", "impulse_noise"],
    "Blur":    ["defocus_blur", "glass_blur", "motion_blur", "zoom_blur"],
    "Weather": ["snow", "frost", "fog", "brightness"],
    "Digital": ["contrast", "elastic_transform",
                "pixelate", "jpeg_compression"]
}
ALL_CORRUPTIONS = [c for cats in CORRUPTION_CATEGORIES.values()
                   for c in cats]
SEVERITIES = [1, 2, 3, 4, 5]

print(f"\n✅ ImageNet-C protocol:")
print(f"   Categories: {list(CORRUPTION_CATEGORIES.keys())}")
print(f"   Total corruptions: {len(ALL_CORRUPTIONS)}")
print(f"   Severities: {SEVERITIES}")

# --- 3.6 CRATTT Hyperparameters ---
# Fixed values from Chapter 3 pilot analysis
# Do not change these for primary results
# Use ablation blocks for sensitivity analysis
CRATTT_PARAMS = {
    "alpha":          0.4,   # DINO weight in Rjoint
    "beta":           0.6,   # Oracle weight in Rjoint
    "tau":            0.25,  # Fixed verification threshold
    "dino_text_thr":  0.12,  # GroundingDINO text threshold (permissive)
    "yolo_conf":      0.12,  # YOLO confidence (matched to DINO)
    "max_regions":    15,    # BARON max region crops
    "region_size":    (224, 224),  # CLIP input size
}
print(f"\n✅ CRATTT hyperparameters locked:")
for k, v in CRATTT_PARAMS.items():
    print(f"   {k}: {v}")

# --- 3.7 Evaluation Settings ---
EVAL_PARAMS = {
    "num_images":   20,
    "num_pilot":    5,
    "save_dir":     DIRS["results"],
    "fig_dir":      DIRS["figures"],
    "table_dir":    DIRS["tables"],
    "ckpt_dir":     DIRS["checkpoints"],
}
print(f"\n✅ Evaluation parameters:")
for k, v in EVAL_PARAMS.items():
    print(f"   {k}: {v}")

print("\n" + "="*50)
print("BLOCK 3 COMPLETE — Constants defined")
print("="*50)

✅ COCO_MAP loaded: 80 categories
loading annotations into memory...
Done (t=0.50s)
creating index...
index created!
✅ COCO_MAP verified: all 80 IDs match official annotations
✅ YOLO-World classes set: 80 COCO categories

Generating per-class CLIP text embeddings...
✅ CLIP text embeddings: torch.Size([80, 512])
   Expected: torch.Size([80, 512])

   Similarity checks (all must be < 0.98):
   person vs car:   0.8450
   person vs dog:   0.8424
   car vs chair:    0.8068

   ✅ All embeddings are distinct

✅ ImageNet-C protocol:
   Categories: ['Noise', 'Blur', 'Weather', 'Digital']
   Total corruptions: 15
   Severities: [1, 2, 3, 4, 5]

✅ CRATTT hyperparameters locked:
   alpha: 0.4
   beta: 0.6
   tau: 0.25
   dino_text_thr: 0.12
   yolo_conf: 0.12
   max_regions: 15
   region_size: (224, 224)

✅ Evaluation parameters:
   num_images: 20
   num_pilot: 5
   save_dir: /kaggle/working/results
   fig_dir: /kaggle/working/figures
   table_dir: /kaggle/working/tables
   ckpt_dir: /kaggle/workin

In [5]:
import requests
try:
    r = requests.get("https://huggingface.co", timeout=10)
    print(f"✅ Network OK: {r.status_code}")
except Exception as e:
    print(f"❌ Network issue: {e}")

✅ Network OK: 200


In [6]:
# ============================================================
# BLOCK 4: Data Loading & COCO Setup
# Loads 20 COCO validation images into memory once.
# Verifies ground truth annotations are accessible.
# ============================================================

import os
import glob
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

# --- 4.1 Load Image File Paths ---
image_files = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.jpg")))

assert len(image_files) > 0, \
    f"No images found at {IMAGE_DIR} — check dataset path"

# Take the first NUM_IMAGES for evaluation
image_files = image_files[:EVAL_PARAMS["num_images"]]
print(f"✅ Found {len(image_files)} images for evaluation")

# --- 4.2 Build Image ID Map ---
# COCO image IDs are encoded in the filename e.g. 000000000139.jpg → 139
img_id_map = {
    os.path.basename(f): int(os.path.basename(f).split('.')[0])
    for f in image_files
}
coco_img_ids = list(img_id_map.values())
print(f"✅ Image ID map built: {len(img_id_map)} entries")
print(f"   First 3 IDs: {coco_img_ids[:3]}")

# --- 4.3 Pre-load Images Into Memory ---
# Avoids repeated disk reads during the benchmark loops
print(f"\nPre-loading {len(image_files)} images into memory...")
loaded_images = {}

for img_path in tqdm(image_files, desc="Loading"):
    img_array = np.array(Image.open(img_path).convert("RGB"))
    loaded_images[img_path] = img_array

# Memory estimate
sample_shape = next(iter(loaded_images.values())).shape
total_mb = sum(
    img.nbytes for img in loaded_images.values()
) / 1e6

print(f"✅ All images loaded")
print(f"   Sample shape: {sample_shape}")
print(f"   Total memory: {total_mb:.1f} MB")

# --- 4.4 Verify COCO Annotations ---
# Check that ground truth boxes exist for our image IDs
print(f"\nVerifying COCO annotations...")
missing_annotations = []

for img_id in coco_img_ids:
    ann_ids = coco_gt.getAnnIds(imgIds=img_id)
    if len(ann_ids) == 0:
        missing_annotations.append(img_id)

if missing_annotations:
    print(f"⚠️  {len(missing_annotations)} images have no annotations: "
          f"{missing_annotations}")
else:
    print(f"✅ All {len(coco_img_ids)} images have ground truth annotations")

# --- 4.5 Annotation Statistics ---
# Useful context for interpreting mAP results
total_gt_boxes = 0
category_counts = {}

for img_id in coco_img_ids:
    ann_ids = coco_gt.getAnnIds(imgIds=img_id)
    anns = coco_gt.loadAnns(ann_ids)
    total_gt_boxes += len(anns)
    
    for ann in anns:
        cat_name = coco_gt.loadCats(ann['category_id'])[0]['name']
        category_counts[cat_name] = category_counts.get(cat_name, 0) + 1

# Top 10 most frequent categories in our evaluation set
top_cats = sorted(category_counts.items(), 
                  key=lambda x: x[1], reverse=True)[:10]

print(f"\n--- Ground Truth Statistics ---")
print(f"Total GT boxes across {len(image_files)} images: {total_gt_boxes}")
print(f"Mean GT boxes per image: {total_gt_boxes/len(image_files):.1f}")
print(f"\nTop 10 categories in evaluation set:")
for cat, count in top_cats:
    print(f"   {cat:<20} {count:>4} instances")

# --- 4.6 Save Dataset Manifest ---
import json

manifest = {
    "num_images": len(image_files),
    "image_ids": coco_img_ids,
    "total_gt_boxes": total_gt_boxes,
    "mean_gt_per_image": round(total_gt_boxes / len(image_files), 2),
    "top_categories": dict(top_cats)
}

manifest_path = os.path.join(EVAL_PARAMS["save_dir"], "dataset_manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\n✅ Dataset manifest saved: {manifest_path}")

print("\n" + "="*50)
print("BLOCK 4 COMPLETE — Data loaded and verified")
print("="*50)

✅ Found 20 images for evaluation
✅ Image ID map built: 20 entries
   First 3 IDs: [139, 285, 632]

Pre-loading 20 images into memory...


Loading:   0%|          | 0/20 [00:00<?, ?it/s]

✅ All images loaded
   Sample shape: (426, 640, 3)
   Total memory: 16.6 MB

Verifying COCO annotations...
✅ All 20 images have ground truth annotations

--- Ground Truth Statistics ---
Total GT boxes across 20 images: 143
Mean GT boxes per image: 7.2

Top 10 categories in evaluation set:
   person                 54 instances
   book                   16 instances
   car                     8 instances
   chair                   6 instances
   vase                    4 instances
   potted plant            3 instances
   tv                      3 instances
   teddy bear              3 instances
   handbag                 3 instances
   backpack                3 instances

✅ Dataset manifest saved: /kaggle/working/results/dataset_manifest.json

BLOCK 4 COMPLETE — Data loaded and verified
